In [1]:
#import libraries 

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyClassifier, DummyRegressor
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('workflow_engineered.csv')
# Recreate categorical columns
cat_cols = ['Process_Name', 'Task_Type', 'Priority_Level', 'Department', 'Approval_Level']
for col in cat_cols:
    df[col] = df[col].astype('category')

In [5]:
# Define features and targets
# For all models, i will use a common set of features (can be refined later)


feature_cols = ['Estimated_Time_Minutes', 'Employee_Workload', 
                'start_hour', 'start_dayofweek', 'start_month_num'] + cat_cols

X = df[feature_cols].copy()
y_delay = df['Delay_Flag']
y_duration = df['Actual_Time_Minutes']
y_cost = df['Cost_Per_Task']


In [6]:
# Preprocessing: one-hot encode categoricals, scale numeric 


numeric_features = ['Estimated_Time_Minutes', 'Employee_Workload', 'start_hour', 'start_dayofweek', 'start_month_num']
categorical_features = cat_cols

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


In [8]:
# Split data (use same split for all tasks) 

X_train, X_test, y_delay_train, y_delay_test = train_test_split(X, y_delay, test_size=0.2, random_state=42, stratify=y_delay)
_, _, y_dur_train, y_dur_test = train_test_split(X, y_duration, test_size=0.2, random_state=42)
_, _, y_cost_train, y_cost_test = train_test_split(X, y_cost, test_size=0.2, random_state=42)
 

In [9]:
# 1. Delay Prediction (Classification) 

print("="*50)
print("Delay Prediction Models")
print("="*50)

models_class = {
    'Dummy': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss')
}

results_class = {}
for name, model in models_class.items():
    pipeline = Pipeline([('prep', preprocessor), ('clf', model)])
    pipeline.fit(X_train, y_delay_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:,1] if hasattr(pipeline, 'predict_proba') else None
    acc = accuracy_score(y_delay_test, y_pred)
    prec = precision_score(y_delay_test, y_pred)
    rec = recall_score(y_delay_test, y_pred)
    f1 = f1_score(y_delay_test, y_pred)
    auc = roc_auc_score(y_delay_test, y_proba) if y_proba is not None else np.nan
    results_class[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc}
    print(f"{name} -> Accuracy: {acc:.3f}, F1: {f1:.3f}, AUC: {auc:.3f}")

Delay Prediction Models
Dummy -> Accuracy: 0.950, F1: 0.974, AUC: 0.500
Logistic Regression -> Accuracy: 0.950, F1: 0.974, AUC: 0.564
Decision Tree -> Accuracy: 0.880, F1: 0.936, AUC: 0.482
Random Forest -> Accuracy: 0.950, F1: 0.974, AUC: 0.543
Gradient Boosting -> Accuracy: 0.950, F1: 0.974, AUC: 0.501
XGBoost -> Accuracy: 0.946, F1: 0.972, AUC: 0.464


In [10]:
# 2. Duration Prediction (Regression) 

print("\n" + "="*50)
print("Duration Prediction Models")
print("="*50)

models_reg = {
    'Dummy': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42)
}

results_dur = {}
for name, model in models_reg.items():
    pipeline = Pipeline([('prep', preprocessor), ('reg', model)])
    pipeline.fit(X_train, y_dur_train)
    y_pred = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_dur_test, y_pred))
    mae = mean_absolute_error(y_dur_test, y_pred)
    r2 = r2_score(y_dur_test, y_pred)
    results_dur[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name} -> RMSE: {rmse:.1f}, MAE: {mae:.1f}, R2: {r2:.3f}")
 


Duration Prediction Models
Dummy -> RMSE: 72.5, MAE: 61.4, R2: -0.000
Linear Regression -> RMSE: 73.0, MAE: 61.9, R2: -0.014
Decision Tree -> RMSE: 106.1, MAE: 87.0, R2: -1.142
Random Forest -> RMSE: 74.9, MAE: 63.0, R2: -0.067
Gradient Boosting -> RMSE: 73.0, MAE: 61.4, R2: -0.014
XGBoost -> RMSE: 83.5, MAE: 68.6, R2: -0.326


In [11]:
# 3. Cost Prediction (Regression) 

print("\n" + "="*50)
print("Cost Prediction Models")
print("="*50)

results_cost = {}
for name, model in models_reg.items():
    pipeline = Pipeline([('prep', preprocessor), ('reg', model)])
    pipeline.fit(X_train, y_cost_train)
    y_pred = pipeline.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_cost_test, y_pred))
    mae = mean_absolute_error(y_cost_test, y_pred)
    r2 = r2_score(y_cost_test, y_pred)
    results_cost[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.3f}") 


Cost Prediction Models
Dummy -> RMSE: 129.23, MAE: 111.24, R2: -0.002
Linear Regression -> RMSE: 129.91, MAE: 111.80, R2: -0.012
Decision Tree -> RMSE: 187.21, MAE: 153.48, R2: -1.102
Random Forest -> RMSE: 134.03, MAE: 113.59, R2: -0.078
Gradient Boosting -> RMSE: 132.52, MAE: 113.51, R2: -0.053
XGBoost -> RMSE: 149.63, MAE: 126.20, R2: -0.343


In [ ]:
# 4. SLA Compliance (same as Delay_Flag, so we skip duplication) 

# Save the best models (using XGBoost for all for now)
best_delay_model = Pipeline([('prep', preprocessor), ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss'))])
best_delay_model.fit(X_train, y_delay_train)
joblib.dump(best_delay_model, 'delay_model.pkl')

best_duration_model = Pipeline([('prep', preprocessor), ('reg', xgb.XGBRegressor(random_state=42))])
best_duration_model.fit(X_train, y_dur_train)
joblib.dump(best_duration_model, 'duration_model.pkl')

best_cost_model = Pipeline([('prep', preprocessor), ('reg', xgb.XGBRegressor(random_state=42))])
best_cost_model.fit(X_train, y_cost_train)
joblib.dump(best_cost_model, 'cost_model.pkl')

print("\nModels saved.")